In [27]:
import pyhid_usb_relay
relay = pyhid_usb_relay.find()

In [41]:
relay.toggle_state(5)

In [25]:
relay.get_state(8)

False

In [44]:
import usb.core

dev = usb.core.find(idVendor=0x16c0, idProduct=0x05df, find_all=True)
for d in dev:
    print(f"Bus: {d.bus}, Address: {d.address}")

Bus: 2, Address: 17
Bus: 2, Address: 16


In [52]:
import hid

VID = 0x16c0
PID = 0x05df

devices = list(hid.enumerate(VID, PID))

for i, dev in enumerate(devices):
    print(f"[{i}] Path: {dev['path']}")
    print(f"    Serial: {dev['serial_number']}")
    print(f"    Manufacturer: {dev.get('manufacturer_string')}")
    print(f"    Product: {dev.get('product_string')}")
    print("-" * 30)

[0] Path: b'\\\\?\\HID#VID_16C0&PID_05DF#8&37925b62&0&0000#{4d1e55b2-f16f-11cf-88cb-001111000030}'
    Serial: 
    Manufacturer: www.dcttech.com
    Product: USBRelay8
------------------------------
[1] Path: b'\\\\?\\HID#VID_16C0&PID_05DF#9&142c2258&0&0000#{4d1e55b2-f16f-11cf-88cb-001111000030}'
    Serial: 
    Manufacturer: www.dcttech.com
    Product: USBRelay8
------------------------------


In [ ]:
import usb.core
import usb.util
import time
import platform

VENDOR_ID = 0x16c0
PRODUCT_ID = 0x05df
DEVICE_1 = dict(idVendor=VENDOR_ID, idProduct=PRODUCT_ID, bus=2, address=17)
DEVICE_2 = dict(idVendor=VENDOR_ID, idProduct=PRODUCT_ID, bus=2, address=16)

SET_REPORT = 0x9
REPORT_TYPE_FEATURE = 3


class USBRelay8:
    def __init__(self, **kwargs):
        self._dev = usb.core.find(**kwargs)
        if self._dev is None:
            raise ValueError("未找到设备")
        # Windows 上可能需要设置配置
        try:
            self._dev.set_configuration()
        except usb.core.USBError:
            pass  # 已经配置过

    def set_output(self, relay_num, state):
        cmd_byte = 0xFF if state else 0xFD
        payload = [cmd_byte, relay_num]

        # Windows 关键修复：填充到 64 字节
        if platform.system() == 'Windows':
            payload = payload + [0] * (64 - len(payload))

        self._dev.ctrl_transfer(
            usb.util.CTRL_TYPE_CLASS | usb.util.CTRL_RECIPIENT_DEVICE | usb.util.ENDPOINT_OUT,
            SET_REPORT,
            (REPORT_TYPE_FEATURE << 8) | 0,
            0,
            payload,
            timeout=1000
        )

    def get_output(self, relay_num):
        resp = self._dev.ctrl_transfer(
            usb.util.CTRL_TYPE_CLASS | usb.util.CTRL_RECIPIENT_DEVICE | usb.util.ENDPOINT_IN,
            0x1,  # GET_REPORT
            (REPORT_TYPE_FEATURE << 8) | 0,
            0,
            8,
            timeout=1000
        )
        return bool(resp[7] & (1 << (relay_num - 1)))

    def get_all_states(self):
        return [self.get_output(i) for i in range(1, 9)]

    def close(self):
        usb.util.dispose_resources(self._dev)




两个设备均已连接
出错: [Errno 5] Input/Output Error


In [ ]:
if __name__ == "__main__":
    relay1 = USBRelay8(**DEVICE_1)
    relay2 = USBRelay8(**DEVICE_2)

    relay1.set_output(1, True)
    time.sleep(0.1)
    print("设备1:", relay1.get_all_states())

    relay2.set_output(3, True)
    time.sleep(0.1)
    print("设备2:", relay2.get_all_states())

    relay1.close()
    relay2.close()

出错: [Errno 5] Input/Output Error
